# 🏦 FX Treasury Operations Simulator
### End-to-end treasury desk simulation — FX, Money Markets, OTC Derivatives

| Module | Coverage |
|---|---|
| **FX Spot & Forward** | Live rates via yfinance, Interest Rate Parity pricing |
| **Money Markets** | T-Bills, CP, CD — discount yield → price conversion |
| **OTC Derivatives** | FX Forward book — Mark-to-Market valuation |
| **SOFR Benchmark** | Risk-free rate via FRED API |
| **Dashboard** | 4-chart interactive Plotly dashboard |
| **Ops Report** | Daily summary + multi-sheet Excel export |

In [ ]:
# ============================================================
# CELL 1 — Install & Import
# ============================================================
!pip install -q yfinance pandas-datareader plotly openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import yfinance as yf
import pandas_datareader as pdr
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded')

## Module 1 — FX Spot & Forward Pricing

In [ ]:
# ============================================================
# CELL 2 — Pull Live FX Spot Rates
# ============================================================
print('📡 Pulling live FX spot rates via yfinance...')

pairs = ['EURUSD=X', 'GBPUSD=X', 'USDJPY=X', 'AUDUSD=X']
raw   = yf.download(pairs, period='1y', interval='1d', auto_adjust=True)
fx_data = raw['Close'].copy()
fx_data.columns = ['AUD/USD', 'EUR/USD', 'GBP/USD', 'USD/JPY']
fx_data = fx_data.dropna()

latest_spots = {
    'EUR/USD': float(fx_data['EUR/USD'].iloc[-1]),
    'GBP/USD': float(fx_data['GBP/USD'].iloc[-1]),
    'USD/JPY': float(fx_data['USD/JPY'].iloc[-1]),
    'AUD/USD': float(fx_data['AUD/USD'].iloc[-1]),
}

print('\n📌 Latest FX Spot Rates:')
for pair, rate in latest_spots.items():
    print(f'   {pair:<10}: {rate:.4f}')
print(f'\n✅ {len(fx_data)} trading days loaded')

In [ ]:
# ============================================================
# CELL 3 — Forward Rate Calculation (Interest Rate Parity)
# ============================================================
# F = S × (1 + r_domestic × days/360) / (1 + r_foreign × days/360)

def calc_forward_rate(spot, r_domestic, r_foreign, days):
    """Calculate FX forward rate using Covered Interest Rate Parity."""
    return spot * (1 + r_domestic * days / 360) / (1 + r_foreign * days / 360)

# Interest rate assumptions (approximate central bank rates)
rates = {
    'USD': 0.053,   # Fed Funds
    'EUR': 0.040,   # ECB
    'GBP': 0.052,   # Bank of England
    'JPY': 0.001,   # Bank of Japan
    'AUD': 0.043,   # RBA
}

# Generate forward curves for EUR/USD (1M, 2M, 3M, 6M, 1Y)
tenors = {'1M': 30, '2M': 60, '3M': 90, '6M': 180, '1Y': 360}

for tenor, days in tenors.items():
    fx_data[f'EUR/USD_Fwd_{tenor}'] = calc_forward_rate(
        fx_data['EUR/USD'], r_domestic=rates['USD'], r_foreign=rates['EUR'], days=days
    )
    fx_data[f'GBP/USD_Fwd_{tenor}'] = calc_forward_rate(
        fx_data['GBP/USD'], r_domestic=rates['USD'], r_foreign=rates['GBP'], days=days
    )

# Forward points (pips) = Forward - Spot
print('📌 EUR/USD Forward Rates (latest):')
spot_eurusd = latest_spots['EUR/USD']
print(f'   Spot        : {spot_eurusd:.4f}')
for tenor, days in tenors.items():
    fwd = calc_forward_rate(spot_eurusd, rates['USD'], rates['EUR'], days)
    pts = (fwd - spot_eurusd) * 10000
    print(f'   Fwd {tenor:<4}    : {fwd:.4f}  ({pts:+.1f} pips)')

print('\n✅ Forward curves generated')

## Module 2 — Money Market Instruments

In [ ]:
# ============================================================
# CELL 4 — Pull SOFR (Risk-Free Rate Benchmark)
# ============================================================
print('📡 Pulling SOFR from FRED API...')

try:
    sofr = pdr.get_data_fred('SOFR', start='2023-01-01')
    sofr.columns = ['SOFR']
    sofr = sofr.dropna()
    print(f'✅ SOFR loaded — {len(sofr)} observations')
    print(f'   Latest SOFR : {sofr["SOFR"].iloc[-1]:.3f}%')
    print(f'   As of       : {sofr.index[-1].date()}')
except Exception as e:
    print(f'⚠️  FRED API unavailable ({e}) — using simulated SOFR')
    dates = pd.date_range('2023-01-01', periods=400, freq='B')
    sofr  = pd.DataFrame({'SOFR': np.linspace(4.3, 5.3, 400) + np.random.normal(0, 0.02, 400)}, index=dates)

In [ ]:
# ============================================================
# CELL 5 — Money Market Portfolio Pricing
# ============================================================

def tbill_price(face_value, discount_rate, days):
    """Convert T-Bill discount yield to price."""
    return face_value * (1 - discount_rate * days / 360)

def tbill_yield(face_value, price, days):
    """Convert T-Bill price to bond-equivalent yield."""
    return (face_value - price) / price * (365 / days)

def cp_price(face_value, discount_rate, days):
    """Commercial Paper price (same as T-Bill, discount basis)."""
    return tbill_price(face_value, discount_rate, days)

# MM Portfolio
mm_portfolio = pd.DataFrame({
    'instrument'       : ['T-Bill 90d',  'T-Bill 180d', 'CP 30d',    'CD 60d',    'T-Bill 270d'],
    'type'             : ['T-Bill',       'T-Bill',      'CP',        'CD',        'T-Bill'],
    'face_value'       : [1_000_000,      500_000,       750_000,     250_000,     1_000_000],
    'discount_rate'    : [0.052,          0.054,         0.056,       0.053,       0.055],
    'days_to_maturity' : [90,             180,           30,          60,          270],
    'counterparty'     : ['US Treasury',  'US Treasury', 'Corp X',    'Bank Y',    'US Treasury'],
})

mm_portfolio['price']           = mm_portfolio.apply(
    lambda r: tbill_price(r.face_value, r.discount_rate, r.days_to_maturity), axis=1
)
mm_portfolio['discount_amount'] = mm_portfolio['face_value'] - mm_portfolio['price']
mm_portfolio['bey']             = mm_portfolio.apply(
    lambda r: tbill_yield(r.face_value, r.price, r.days_to_maturity), axis=1
)  # Bond-Equivalent Yield
mm_portfolio['sofr_spread_bps'] = (mm_portfolio['discount_rate'] - sofr['SOFR'].iloc[-1]/100) * 10000

print('📌 Money Market Portfolio:')
print(mm_portfolio[['instrument','face_value','price','discount_rate','bey','days_to_maturity','sofr_spread_bps']]
      .to_string(index=False))
print(f'\n   Total Face Value  : ${mm_portfolio["face_value"].sum():>15,.0f}')
print(f'   Total Price       : ${mm_portfolio["price"].sum():>15,.2f}')
print(f'   Total Discount    : ${mm_portfolio["discount_amount"].sum():>15,.2f}')
print(f'   Avg Discount Rate : {mm_portfolio["discount_rate"].mean():.3%}')
print(f'   Avg BEY           : {mm_portfolio["bey"].mean():.3%}')
print('\n✅ MM portfolio priced')

## Module 3 — OTC FX Forward Book (Mark-to-Market)

In [ ]:
# ============================================================
# CELL 6 — OTC Forward Book MTM
# ============================================================

forward_book = pd.DataFrame({
    'counterparty'     : ['Corp A',    'Corp B',    'Hedge Fund C', 'Corp D',    'Bank E',    'Fund F'],
    'pair'             : ['EUR/USD',   'GBP/USD',   'USD/JPY',      'EUR/USD',   'GBP/USD',   'AUD/USD'],
    'direction'        : ['BUY',       'SELL',      'BUY',          'SELL',      'BUY',       'SELL'],
    'notional_usd'     : [5_000_000,   2_000_000,   10_000_000,     3_000_000,   4_000_000,   1_500_000],
    'contracted_rate'  : [1.0850,      1.2650,      149.50,         1.0920,      1.2720,      0.6550],
    'days_remaining'   : [45,          30,           60,             90,          15,           120],
    'r_domestic'       : [0.053]*6,
    'r_foreign'        : [0.040,       0.052,       0.001,          0.040,       0.052,       0.043],
})

def fx_forward_mtm(row):
    spot = latest_spots.get(row['pair'], row['contracted_rate'])
    fwd  = calc_forward_rate(spot, row['r_domestic'], row['r_foreign'], row['days_remaining'])
    # MTM = notional × (current_fwd - contracted_rate) / current_fwd
    if row['direction'] == 'BUY':
        mtm = row['notional_usd'] * (fwd - row['contracted_rate']) / fwd
    else:  # SELL
        mtm = row['notional_usd'] * (row['contracted_rate'] - fwd) / fwd
    return round(mtm, 2)

forward_book['current_spot']    = forward_book['pair'].map(latest_spots)
forward_book['current_fwd']     = forward_book.apply(
    lambda r: round(calc_forward_rate(
        latest_spots.get(r['pair'], r['contracted_rate']),
        r['r_domestic'], r['r_foreign'], r['days_remaining']
    ), 4), axis=1
)
forward_book['mtm_pnl_usd']     = forward_book.apply(fx_forward_mtm, axis=1)
forward_book['mtm_status']      = forward_book['mtm_pnl_usd'].apply(
    lambda x: '🟢 GAIN' if x > 0 else '🔴 LOSS'
)
forward_book['alert']           = forward_book['mtm_pnl_usd'].apply(
    lambda x: '⚠️ REVIEW' if abs(x) > 50_000 else ''
)

print('📋 OTC Forward Book — Mark-to-Market Report:')
print(forward_book[['counterparty','pair','direction','notional_usd',
                     'contracted_rate','current_fwd','mtm_pnl_usd','mtm_status','alert']]
      .to_string(index=False))
print(f'\n   Net MTM P&L       : ${forward_book["mtm_pnl_usd"].sum():>12,.2f}')
print(f'   Total Notional    : ${forward_book["notional_usd"].sum():>12,.0f}')
print(f'   Contracts in Gain : {(forward_book["mtm_pnl_usd"]>0).sum()}')
print(f'   Contracts in Loss : {(forward_book["mtm_pnl_usd"]<0).sum()}')
print(f'   Alerts (>$50K)    : {(forward_book["alert"]!="").sum()}')
print('\n✅ OTC MTM complete')

## Module 4 — Dashboard & Reporting

In [ ]:
# ============================================================
# CELL 7 — Interactive Dashboard (4 Charts)
# ============================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        '📈 EUR/USD — Spot vs Forward Curve (90d)',
        '📊 MM Portfolio — Maturity Ladder',
        '💹 OTC Forward Book — MTM P&L per Counterparty',
        '📉 SOFR Rate Trend (Risk-Free Benchmark)',
    ]
)

# --- Chart 1: EUR/USD Spot vs Forward ---
sample = fx_data[['EUR/USD','EUR/USD_Fwd_1M','EUR/USD_Fwd_3M','EUR/USD_Fwd_6M']].dropna().tail(90)
for col, color, name, dash in [
    ('EUR/USD',         '#2196F3', 'Spot',    'solid'),
    ('EUR/USD_Fwd_1M',  '#4CAF50', 'Fwd 1M',  'dot'),
    ('EUR/USD_Fwd_3M',  '#FF9800', 'Fwd 3M',  'dash'),
    ('EUR/USD_Fwd_6M',  '#F44336', 'Fwd 6M',  'dashdot'),
]:
    fig.add_trace(go.Scatter(
        x=sample.index, y=sample[col],
        name=name, mode='lines',
        line=dict(color=color, dash=dash, width=2)
    ), row=1, col=1)

# --- Chart 2: MM Maturity Ladder ---
colors_mm = ['#2196F3' if t=='T-Bill' else '#FF9800' if t=='CP' else '#9C27B0'
             for t in mm_portfolio['type']]
fig.add_trace(go.Bar(
    x=mm_portfolio['instrument'],
    y=mm_portfolio['face_value'] / 1e6,
    name='Face Value (USD M)',
    marker_color=colors_mm,
    text=[f"{d}d" for d in mm_portfolio['days_to_maturity']],
    textposition='auto'
), row=1, col=2)

# --- Chart 3: OTC MTM Bar ---
colors_mtm = ['#4CAF50' if x > 0 else '#F44336' for x in forward_book['mtm_pnl_usd']]
fig.add_trace(go.Bar(
    x=forward_book['counterparty'],
    y=forward_book['mtm_pnl_usd'],
    name='MTM P&L (USD)',
    marker_color=colors_mtm,
    text=[f'${v:,.0f}' for v in forward_book['mtm_pnl_usd']],
    textposition='auto'
), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='grey', row=2, col=1)

# --- Chart 4: SOFR Trend ---
sofr_sample = sofr.dropna().tail(250)
fig.add_trace(go.Scatter(
    x=sofr_sample.index, y=sofr_sample['SOFR'],
    name='SOFR (%)', mode='lines',
    line=dict(color='#00BCD4', width=2),
    fill='tozeroy', fillcolor='rgba(0,188,212,0.1)'
), row=2, col=2)
# Add horizontal line for avg MM rate
avg_mm_rate = mm_portfolio['discount_rate'].mean() * 100
fig.add_hline(y=avg_mm_rate, line_dash='dash', line_color='orange',
              annotation_text=f'Avg MM Rate ({avg_mm_rate:.2f}%)',
              row=2, col=2)

fig.update_layout(
    height=780,
    title_text='🏦 FX Treasury Operations Dashboard',
    title_font_size=18,
    showlegend=True,
    template='plotly_white'
)
fig.update_yaxes(title_text='Rate', row=1, col=1)
fig.update_yaxes(title_text='Face Value (USD M)', row=1, col=2)
fig.update_yaxes(title_text='MTM P&L (USD)', row=2, col=1)
fig.update_yaxes(title_text='Rate (%)', row=2, col=2)

fig.show()
print('✅ Dashboard rendered')

In [ ]:
# ============================================================
# CELL 8 — Forward Curve Term Structure
# ============================================================
# Bonus chart: full forward curve for all pairs at today's rates

tenor_labels = list(tenors.keys())   # ['1M','2M','3M','6M','1Y']
tenor_days   = list(tenors.values()) # [30, 60, 90, 180, 360]

curve_data = {}
pair_config = [
    ('EUR/USD', rates['USD'], rates['EUR']),
    ('GBP/USD', rates['USD'], rates['GBP']),
    ('AUD/USD', rates['USD'], rates['AUD']),
]

fig2 = go.Figure()
for pair, r_d, r_f in pair_config:
    spot   = latest_spots[pair]
    fwds   = [calc_forward_rate(spot, r_d, r_f, d) for d in tenor_days]
    # Normalize to spot = 100 for comparability
    fwds_norm = [f/spot * 100 for f in fwds]
    fig2.add_trace(go.Scatter(
        x=['Spot'] + tenor_labels,
        y=[100] + fwds_norm,
        name=pair, mode='lines+markers',
        marker=dict(size=8)
    ))

fig2.update_layout(
    title='📈 FX Forward Term Structure — All Pairs (Spot = 100)',
    xaxis_title='Tenor',
    yaxis_title='Index (Spot = 100)',
    height=400,
    template='plotly_white'
)
fig2.add_hline(y=100, line_dash='dash', line_color='grey',
               annotation_text='Spot Level')
fig2.show()
print('✅ Term structure chart rendered')

In [ ]:
# ============================================================
# CELL 9 — Daily Operations Summary Report
# ============================================================
from datetime import date

sofr_latest = float(sofr['SOFR'].dropna().iloc[-1])
mm_avg_rate = float(mm_portfolio['discount_rate'].mean() * 100)
spread_bps  = (mm_avg_rate - sofr_latest) * 100

print('=' * 65)
print('         FX TREASURY OPERATIONS — DAILY SUMMARY REPORT')
print(f'         Report Date: {date.today()}')
print('=' * 65)

print('\n[1] FX SPOT RATES')
for pair, rate in latest_spots.items():
    print(f'    {pair:<10}: {rate:.4f}')

print('\n[2] EUR/USD FORWARD CURVE')
print(f'    {"Tenor":<8} {"Rate":>8} {"Pips":>8}')
spot_eur = latest_spots['EUR/USD']
print(f'    {"Spot":<8} {spot_eur:>8.4f} {0:>+8.1f}')
for tenor, days in tenors.items():
    fwd  = calc_forward_rate(spot_eur, rates['USD'], rates['EUR'], days)
    pips = (fwd - spot_eur) * 10000
    print(f'    {tenor:<8} {fwd:>8.4f} {pips:>+8.1f}')

print('\n[3] MONEY MARKET PORTFOLIO')
print(f'    Total Face Value   : ${mm_portfolio["face_value"].sum():>15,.0f}')
print(f'    Total Price        : ${mm_portfolio["price"].sum():>15,.2f}')
print(f'    Total Discount     : ${mm_portfolio["discount_amount"].sum():>15,.2f}')
print(f'    Avg Discount Rate  : {mm_portfolio["discount_rate"].mean():.3%}')
print(f'    Avg BEY            : {mm_portfolio["bey"].mean():.3%}')
print(f'    Shortest Maturity  : {mm_portfolio["days_to_maturity"].min()}d ({mm_portfolio.loc[mm_portfolio["days_to_maturity"].idxmin(),"instrument"]})')

print('\n[4] OTC FORWARD BOOK')
print(f'    Active Contracts   : {len(forward_book)}')
print(f'    Total Notional     : ${forward_book["notional_usd"].sum():>15,.0f}')
print(f'    Net MTM P&L        : ${forward_book["mtm_pnl_usd"].sum():>15,.2f}')
print(f'    Contracts in Gain  : {(forward_book["mtm_pnl_usd"]>0).sum()}')
print(f'    Contracts in Loss  : {(forward_book["mtm_pnl_usd"]<0).sum()}')
print(f'    High-Risk Alerts   : {(forward_book["alert"]!="").sum()}')
if (forward_book['alert']!='').any():
    alerts = forward_book[forward_book['alert']!='']
    for _, row in alerts.iterrows():
        print(f'      ⚠️  {row["counterparty"]} | {row["pair"]} | MTM: ${row["mtm_pnl_usd"]:,.0f}')

print('\n[5] BENCHMARK')
print(f'    SOFR (latest)      : {sofr_latest:.3f}%')
print(f'    MM Portfolio Rate  : {mm_avg_rate:.3f}%')
print(f'    Spread vs SOFR     : {spread_bps:+.1f} bps')

print('\n' + '=' * 65)
print('✅ Report complete.')

In [ ]:
# ============================================================
# CELL 10 — Export Multi-Sheet Excel Ops Report
# ============================================================
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

output_file = 'FX_Treasury_OpsReport.xlsx'

# Build forward curve snapshot
fwd_snapshot = pd.DataFrame({
    'Tenor'      : ['Spot'] + list(tenors.keys()),
    'Days'       : [0]      + list(tenors.values()),
    'EUR/USD'    : [spot_eur] + [calc_forward_rate(spot_eur, rates['USD'], rates['EUR'], d) for d in tenors.values()],
    'GBP/USD'    : [latest_spots['GBP/USD']] + [calc_forward_rate(latest_spots['GBP/USD'], rates['USD'], rates['GBP'], d) for d in tenors.values()],
})
fwd_snapshot['EUR/USD Pips'] = (fwd_snapshot['EUR/USD'] - spot_eur) * 10000

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

    # Sheet 1: FX Rates (last 30 days)
    fx_export = fx_data[['EUR/USD','GBP/USD','USD/JPY','AUD/USD',
                          'EUR/USD_Fwd_1M','EUR/USD_Fwd_3M','EUR/USD_Fwd_6M']].dropna().tail(30).round(4)
    fx_export.index.name = 'Date'
    fx_export.to_excel(writer, sheet_name='FX Rates')

    # Sheet 2: Forward Curve Snapshot
    fwd_snapshot.to_excel(writer, sheet_name='Forward Curve', index=False)

    # Sheet 3: MM Portfolio
    mm_export = mm_portfolio.copy()
    mm_export.to_excel(writer, sheet_name='MM Portfolio', index=False)

    # Sheet 4: OTC Forward Book
    otc_export = forward_book[[
        'counterparty','pair','direction','notional_usd',
        'contracted_rate','current_spot','current_fwd',
        'days_remaining','mtm_pnl_usd','mtm_status','alert'
    ]].copy()
    otc_export.to_excel(writer, sheet_name='OTC Forward Book', index=False)

    # Sheet 5: SOFR Benchmark
    sofr.tail(90).to_excel(writer, sheet_name='SOFR Benchmark')

# Basic formatting pass
wb = openpyxl.load_workbook(output_file)
header_fill = PatternFill(start_color='1F4E79', end_color='1F4E79', fill_type='solid')
header_font = Font(color='FFFFFF', bold=True)

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center')
    for col in ws.columns:
        max_len = max((len(str(c.value)) for c in col if c.value), default=10)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 30)

wb.save(output_file)

print(f'✅ Excel report exported: {output_file}')
print('   Sheets: FX Rates | Forward Curve | MM Portfolio | OTC Forward Book | SOFR Benchmark')

# Auto-download in Colab
from google.colab import files
files.download(output_file)
print('📥 Download triggered')